### 1. Data collection
The first task is to collect the data to analyse later on. The analysis will be conducted on a 100 popular games using player counts, review and Reddit sentiment scores and Twitch viewer counts collected for the duration of 1 year. The first step is selecting the game titles to analyse. For this purpose, 400 candidate games are chosen from the Steam top sellers list.

In [1]:
#Loading the libraries

import pandas as pd
import requests
import os
import csv
import time
from datetime import datetime
from qbittorrentapi import Client
import sys
import re
import locale
from bs4 import BeautifulSoup
import json
import warnings
from dotenv import load_dotenv

In [2]:
candidate_games = []

#Pattern for extracting appid from the logo url from the response
appid_pattern = re.compile(r'/apps/(?P<id>\d+)/')

#Looping through Steam store pages to collect candidate games
for page in range(1, 21):  # Pages 1, 2, and 3
    response = requests.get(
        "https://store.steampowered.com/search/results/",
        params={
            "filter": "topsellers",
            "page": page,
            "cc": "us",
            "l": "english",
            "json": 1
        }
    )

    if response.status_code == 200:
        data = response.json()
        for item in data.get('items', []):
            game_name = item.get('name')
            logo_url = item.get('logo')

            match = appid_pattern.search(logo_url)
            if match:
                appid = int(match.group('id'))
                candidate_games.append({
                    "appid": appid,
                    "name": game_name
                })
            else:
                print("No appid for game:", game_name)
    else:
        print("Error", response.status_code)

print(f"\nGot {len(candidate_games)} games")


Got 500 games


In [3]:
candidate_games_df = pd.DataFrame(candidate_games, columns=["appid", "name"])
candidate_games_df.head(10)

,appid,name
0,2806050,Halo: Campaign Evolved
1,730,Counter-Strike 2
2,219990,Grim Dawn
3,2699230,Grim Dawn - Fangs of Asterkarn
4,1623730,Palworld
5,2424420,Avatar Legends: The Fighting Game
6,2767030,Marvel Rivals
7,4570720,DragonSword : Awakening
8,3440070,Dinoblade
9,4165910,Steam Machine


In [2]:
date = datetime.now().strftime(format="%d-%m-%y")

In [5]:
candidate_games_df.to_csv(f"candidate_games{date}.csv", index=False, quoting=csv.QUOTE_NONNUMERIC)

As can be seen from the preview, some items on the list of candidates are not games, but hardware, demos, beta tests or tools and apps, like the Steam Machine. Some of them might also have been published too recently to get a full 1 year of data. In order to get the final list of 100 games, the candidates have to be filtered.

In [6]:
#Defining a function to parse date strings into datetime objects
def parse_date(date_str):
    formats_to_try = [
        "%d %b %Y",  # Matches '21 Jul 2026' (No comma)
        "%d %b, %Y", # Matches '21 Jul, 2026'
        "%b %d, %Y", # Matches 'Jul 21, 2026'
        "%b %d %Y",  # Matches 'Jul 21 2026'
        "%b %Y",     # Matches 'Jul 2026'
        "%Y",        # Matches just a year '2026'
    ]
    for fmt in formats_to_try:
        try:
            return datetime.strptime(date_str, fmt)
        except ValueError:
            continue
    raise ValueError(f"Unable to parse date string: {date_str}")

#Setting the locale to 'usa' for consistent date parsing
locale.setlocale(locale.LC_TIME, 'usa')

'English_United States.1252'

In [7]:
filtered_games = []

#Iterating through the candidate games and filtering based on release date and type
for item in candidate_games_df.itertuples(index=False):
    app_id = item.appid
    url = f"https://store.steampowered.com/api/appdetails?appids={app_id}"
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        if str(app_id) in data and data[str(app_id)]["success"]:
            print(f"Processing appid {app_id}: {item.name}")
            game_data = data[str(app_id)]["data"]
            release_date = game_data.get("release_date", {}).get("date")
            try:
                release_date_obj = parse_date(release_date.strip())
            except ValueError:
                print(f"Skipping {item.name} with invalid release date: {release_date}")
                continue
            cutoff_date = datetime(2025, 1, 1)
            #Filtering games that are of type 'game', not coming soon, and have a release date after the cutoff date
            if game_data.get("type") == "game" and game_data.get("release_date", {}).get("coming_soon") is False and release_date_obj < cutoff_date:
                filtered_games.append(item)
                print(f"Added {item.name} with release date {release_date_obj.strftime('%Y-%m-%d')}")
        time.sleep(5)  # Sleep for 5 seconds to avoid hitting the API too quickly
    else:
        print(f"Error fetching details for appid {app_id}: {response.status_code}")

print(f"\nFiltered down to {len(filtered_games)} games")
        

Processing appid 2806050: Halo: Campaign Evolved
Processing appid 730: Counter-Strike 2
Added Counter-Strike 2 with release date 2012-08-21
Processing appid 219990: Grim Dawn
Added Grim Dawn with release date 2016-02-25
Processing appid 2699230: Grim Dawn - Fangs of Asterkarn
Processing appid 1623730: Palworld
Processing appid 2424420: Avatar Legends: The Fighting Game
Processing appid 2767030: Marvel Rivals
Added Marvel Rivals with release date 2024-12-05
Processing appid 4570720: DragonSword : Awakening
Processing appid 3440070: Dinoblade
Processing appid 4165910: Steam Machine
Processing appid 1913120: Tears of Metal
Processing appid 1675200: Steam Deck
Processing appid 3722330: Shift At Midnight
Processing appid 3751950: Assassin's Creed Black Flag Resynced
Processing appid 3787240: MARVEL Tōkon: Fighting Souls
Processing appid 230410: Warframe
Added Warframe with release date 2013-03-25
Processing appid 3564740: Where Winds Meet
Processing appid 1938090: Call of Duty®
Added Call o

In [8]:
filtered_games_df = pd.DataFrame(filtered_games, columns=["appid", "name"])
filtered_games_df.to_csv(f"filtered_games_by_metadata{date}.csv", index=False, quoting=csv.QUOTE_NONNUMERIC)

After filtering by metadata, the next step is to ensure there is enough data for analysis from player counts, and reviews. For this purpose, the review histpgraph from Steam will be used to check whether there has been a constistent number of new reviews added during the analysed period. For player counts, the number of datapoints from the SteamCharts website will be used to check whether there is data for all the 365 days.

In [4]:
filtered_games_df = pd.read_csv(f"filtered_games_by_metadata{date}.csv", quoting=csv.QUOTE_NONNUMERIC)
filtered_games_df['appid'] = filtered_games_df['appid'].astype(int)

In [9]:
def filter_histogram(appid):
    
    start_date = datetime(2025, 1, 1)
    end_date = datetime(2025, 12, 31)
    qualify = 0
    
    url = f"https://store.steampowered.com/appreviewhistogram/{appid}"
    params = {
        "json": 1,
        "rollup_type": "year"
    }
    response = requests.get(url, params=params)
    if response.status_code == 200:
        data = response.json()
        results = data.get("results", {})
    else:
        print(f"Error fetching review histogram for appid {appid}: {response.status_code}")

    rollups = results.get("rollups", [])
    for rollup in rollups:
        rollup["date"] = datetime.fromtimestamp(rollup.get("date", 0))
    for rollup in rollups:
        if rollup["date"] < end_date and rollup["date"] >= start_date:
            if rollup["recommendations_up"]+ rollup["recommendations_down"] < 100:
                qualify = 0
            else:
                qualify = 1
    return qualify


In [10]:
def filter_player_counts(appid):
    start_date = datetime(2025, 1, 1)
    end_date = datetime(2025, 12, 31)
    qualify = 0

    url = f"https://steamcharts.com/app/{appid}/chart-data.json"
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        for entry in data:
            date_unix = int(entry[0])
            entry[0] = datetime.fromtimestamp(date_unix / 1000)  # Convert milliseconds to seconds
            if start_date <= entry[0] <= end_date:
                if entry[1] < 1000:
                    qualify = 0
                else:
                    qualify = 1

    return qualify

In order to get and verify Twitch data, the website TwitchStats will be used to get the average number of viewers for each day of 2025 for each game. To get this data, the URL for each game page needs to be determined. As the URLs use both the game name and an ID different from the Steam App ID, the code will use a publicly avaliable file to fetch the ID according to the regular name of the game. Then using that to construct the URL, the viewer data will be fetched and evaluated, to filter out games that do not have enough data to analyse.

In [11]:
games_cache = None

url = "https://twitchstats.net/inc/all.json"
headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome 58.0.3029.110 Safari/537.3"
    }
res = requests.get(url, headers=headers)
if res.status_code == 200:
    games_data = res.json().get("games", [])

    games_cache = {
        g["name"]
        .replace("\\'", "'")
        .replace("\'", "'")
        .strip()
        .lower(): (g["id"], g["name"].replace("\\'", "'").replace("\'", "'"))
        for g in games_data
        if "name" in g and "id" in g
        }
    print(f"Cached {len(games_cache)} games in memory.\n")
else:
    print("Failed to load game database.")

Cached 22331 games in memory.



In [12]:
games_cache_df = pd.DataFrame(games_cache)
games_cache_df.to_csv(f"games cache{date}.csv", index = False, quoting=csv.QUOTE_NONNUMERIC)

In [13]:
def get_twitchstats_id(name):

    clean_target_name = name.strip().lower().replace('®', '').replace('™', '')
    target_id = None
    target_slug = None

    try:
        target_id, target_slug = games_cache.get(clean_target_name)
    except TypeError:
        pass

    if target_id and target_slug:
        return target_id, target_slug
    else:
        return None, None

In [14]:
get_twitchstats_id("Garry's Mod")  # Example usage with a sample game name

('18846', "Garry's Mod")

In [25]:
def get_twitch_views(name):

    twitchstats_id, twitchstats_slug = get_twitchstats_id(name)
    if not twitchstats_id or not twitchstats_slug:
        return None
    url = f"https://twitchstats.net/game/{twitchstats_id}-{twitchstats_slug.replace(' ', '%20')}/lastyear"
    # print(url)

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome 58.0.3029.110 Safari/537.3"
    }
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        html_content = response.text
        soup = BeautifulSoup(html_content, 'html.parser')

        target_script = None
        for script in soup.find_all("script"):
            if script.string and "AmCharts.makeChart" in script.string:
                target_script = script.string
                break

        if not target_script:
            page_text = soup.get_text()

            pattern = r"had ([\d,]+) Average Viewers and ([\d,]+) channels streaming on ([A-Za-z]+ \d{1,2}, \d{4})"
            matches = re.findall(pattern, page_text)

            if not matches:
                return None

            data = []
            for viewers, channels, date_str in matches:
                data.append(
                    {
                        "day": date_str,
                        "value": int(viewers.replace(",", "")),
                        "chans": int(channels.replace(",", "")),
                    }
                )
            df_daily = pd.DataFrame(data)
            df_daily["day"] = pd.to_datetime(df_daily["day"], format="mixed")
            return df_daily

        data_providers = re.findall(r'"dataProvider"\s*:\s*(\[.*?\]),', target_script, re.DOTALL)
        if len(data_providers) >= 4:
            daily_raw = json.loads(data_providers[1])
            df_daily = pd.DataFrame(daily_raw)
            return df_daily
        else:
            print("Could not find the dataProvider for daily stats.")
            return None
    else:
        print(f"Error fetching Twitch stats for game {name}: {response.status_code}")

In [26]:
get_twitch_views("VRChat")  # Example usage with a sample appid

,day,value,chans
0,2025-01-01,9290,46
1,2025-01-02,2683,25
2,2025-01-03,4028,30
3,2025-01-04,3127,31
4,2025-01-05,3514,29
...,...,...,...
360,2025-12-27,4298,28
361,2025-12-28,3122,29
362,2025-12-29,2907,23
363,2025-12-30,2897,25


In [17]:
def filter_twitch_views(name):
    qualify = 0

    twitch_views_df = get_twitch_views(name)
    if twitch_views_df is None:
        return qualify

    start_date = datetime(2025, 1, 1)
    end_date = datetime(2025, 12, 31)


    twitch_views_df['day'] = pd.to_datetime(twitch_views_df['day'], format='mixed')
    filtered_df = twitch_views_df[(twitch_views_df['day'] >= start_date) & (twitch_views_df['day'] <= end_date)]

    if filtered_df.empty:
        print(f"No Twitch view data available for the specified date range for game: {name}")
        return qualify

    total_viewers = filtered_df['value'].sum()
    if total_viewers >= 1000:
        qualify = 1

    return qualify


In [18]:
filter_twitch_views("Dota 2")  # Example usage with a sample appid

1

In [30]:
# Load environment variables from the .env file
load_dotenv()

# Access credentials safely
CLIENT_ID = os.getenv("TWITCH_CLIENT_ID")
CLIENT_SECRET = os.getenv("TWITCH_CLIENT_SECRET")

# Safety check: ensure keys were loaded correctly
if not CLIENT_ID or not CLIENT_SECRET:
    raise ValueError("Missing Twitch API credentials. Please check your .env file!")

def get_igdb_token():
    auth_url = "https://id.twitch.tv/oauth2/token"
    params = {
        'client_id': CLIENT_ID,
        'client_secret': CLIENT_SECRET,
        'grant_type': 'client_credentials'
    }
    response = requests.post(auth_url, params=params, timeout=10)
    response.raise_for_status()
    return response.json()['access_token']

In [28]:
def get_subreddits(name, access_token):

    url = "https://api.igdb.com/v4/games"
    headers = {
        'Client-ID': CLIENT_ID,
        'Authorization': f'Bearer {access_token}'
    }
    
    # Query IGDB and request the 'websites' sub-fields
    body = f'''
    search "{name}";
    fields name, websites.url, websites.category;
    limit 5;
    '''
    
    res = requests.post(url, headers=headers, data=body, timeout=10)
    if res.status_code != 200:
        print(f"IGDB Error {res.status_code}: {res.text}")
        return []

    games = res.json()
    extracted_subreddits = []

    for game in games:
        websites = game.get('websites', [])
        for site in websites:
            site_url = site.get('url', '')
            category = site.get('category')
            
            # Category 14 is Reddit, or fallback to URL string check
            if category == 14 or "reddit.com/r/" in site_url.lower():
                clean_url = site_url.rstrip('/')
                sub_name = clean_url.split('/r/')[-1].split('/')[0]
                
                if sub_name and sub_name not in extracted_subreddits:
                    extracted_subreddits.append(sub_name)

    return extracted_subreddits

In [31]:
def get_post_counts(subreddit_name):
    url = "https://arctic-shift.photon-reddit.com/api/time_series"
    
    # Unix timestamps for the start and end of 2025
    start_date = 1735689600
    end_date   = 1767225599  
    
    params = {
        'key': f'r/{subreddit_name}/posts/count',
        'precision': 'month',
        'after': start_date,
        'before': end_date
    }
    
    headers = {
        'User-Agent': 'ThesisDataCollector/1.0 (academic research)'
    }

    try:
        response = requests.get(url, params=params, headers=headers, timeout=10)
        
        if response.status_code == 200:
            data = response.json().get('data', [])
            
            if not data:
                print(f"No 2025 data found for r/{subreddit_name}")
                return None

            # Parse into a clean Pandas DataFrame
            df = pd.DataFrame(data)
            df['month'] = pd.to_datetime(df['date'], unit='s').dt.strftime('%Y-%m')
            df = df[['month', 'value']].rename(columns={'value': 'post_count'})
            
            return df

        else:
            print(f"HTTP Error {response.status_code} for r/{subreddit_name}")
            return None

    except Exception as e:
        print(f"Request failed for r/{subreddit_name}: {e}")
        return None

In [32]:
def filter_reddit(name, token):
    qualify = 0
    game_subreddits = get_subreddits(name, token)

    for subreddit in game_subreddits:
        df = get_post_counts(subreddit)
        if df is not None and not df.empty:
            if df['post_count'].min() < 100:
                game_subreddits.remove(subreddit)

    if game_subreddits:
        qualify = 1
        
    return qualify
    

In [33]:
token = get_igdb_token()
filter_reddit("Disney Dreamlight Valley", token)

1

In [27]:
token = get_igdb_token()
filtered_games_reviews = []
filtering_result = []
for game in filtered_games_df.itertuples(index=False):
    appid = game.appid
    name = game.name
    histogram_verify = filter_histogram(appid)
    player_counts_verify = filter_player_counts(appid)
    twitch_views_verify = filter_twitch_views(name)
    reddit_verify = filter_reddit(name, token)
    filtering_result.append({
        "appid": appid,
        "name": name,
        "histogram": histogram_verify,
        "player_counts": player_counts_verify,
        "twitch": twitch_views_verify,
        "reddit": reddit_verify
    })
    if histogram_verify == 1 and player_counts_verify == 1 and twitch_views_verify == 1 and reddit_verify == 1:
        filtered_games_reviews.append(game)
        print("Appended game", appid, name)
    else:
        print(f"Skipped game {appid}, {name}")
    time.sleep(1)  # Sleep for 1 second to avoid hitting the API too quickly

print(f"\nFiltered down to {len(filtered_games_reviews)} games")


Skipped game 730, Counter-Strike 2
Skipped game 219990, Grim Dawn
Appended game 2767030 Marvel Rivals
Appended game 230410 Warframe
Skipped game 1938090, Call of Duty®
Appended game 1172470 Apex Legends™
Skipped game 381210, Dead by Daylight
Skipped game 1149460, ICARUS
Appended game 304390 FOR HONOR™
Appended game 582660 Black Desert
Appended game 252490 Rust
Skipped game 1974050, Torchlight: Infinite
Appended game 39210 FINAL FANTASY XIV Online
Appended game 216150 MapleStory
Skipped game 2131680, METAL GEAR &amp; METAL GEAR 2: Solid Snake
Appended game 1085660 Destiny 2
Skipped game 2357570, Overwatch®
Appended game 1361210 Warhammer 40,000: Darktide
Skipped game 2141910, Magic: The Gathering Arena
Skipped game 359550, Tom Clancy's Rainbow Six Siege
Appended game 236390 War Thunder
Appended game 570 Dota 2
Skipped game 2139460, Once Human
Appended game 2840770 Avatar: Frontiers of Pandora™
Skipped game 440, Team Fortress 2
Appended game 1245620 ELDEN RING
Skipped game 2183900, Warha

In [28]:
df_filtering_result = pd.DataFrame(filtering_result)
df_filtered_games_reviews = pd.DataFrame(filtered_games_reviews)

In [29]:
df_filtered_games_reviews.to_csv(f"filtered_games_by_data{date}.csv", index = False, quoting=csv.QUOTE_NONNUMERIC)
df_filtering_result.to_csv(f"filtering_result{date}.csv", index = False, quoting=csv.QUOTE_NONNUMERIC)

----------------------------------------------------------

### Collecting data for the selected filtered games

In [3]:
df_filtered_games_reviews = pd.read_csv(f"filtered_games_by_data24-07-26.csv", quoting=csv.QUOTE_NONNUMERIC)
df_filtered_games_reviews['appid'] = df_filtered_games_reviews['appid'].astype(int)

In [34]:
subreddits_list = []

for idx, game in df_filtered_games_reviews.iterrows():
    print(f"Searching subreddits for {game['name']}...")
    subs = get_subreddits(game['name'], token)

    subreddits_list.append({
        "appid": game["appid"],
        "game_name": game["name"],
        "subreddits": subs
    })

subreddits = pd.DataFrame(subreddits_list)

Searching subreddits for Marvel Rivals...
Searching subreddits for Warframe...
Searching subreddits for Apex Legends™...
Searching subreddits for FOR HONOR™...
Searching subreddits for Black Desert...
Searching subreddits for Rust...
Searching subreddits for FINAL FANTASY XIV Online...
Searching subreddits for MapleStory...
Searching subreddits for Destiny 2...
Searching subreddits for Warhammer 40,000: Darktide...
Searching subreddits for War Thunder...
Searching subreddits for Dota 2...
Searching subreddits for Avatar: Frontiers of Pandora™...
Searching subreddits for ELDEN RING...
Searching subreddits for HELLDIVERS™ 2...
Searching subreddits for The Sims™ 4...
Searching subreddits for Cyberpunk 2077...
Searching subreddits for Limbus Company...
Searching subreddits for Path of Exile...
Searching subreddits for Resident Evil 4...
Searching subreddits for Street Fighter™ 6...
Searching subreddits for ARK: Survival Ascended...
Searching subreddits for Ready or Not...
Searching subredd

In [35]:
subreddits.to_csv(f'game_subreddits{date}.csv', index=False, encoding='utf-8')

In [4]:
import csv
import os

# 1. Paths
TORRENT_DOWNLOAD_DIR = (
    r"C:\Users\magda\My stuff\Projekty\get data magisterka\reddit_new\reddit"
)
CSV_PATH = "game_subreddits29-07-26.csv"  # Replace with your CSV file name

# 2. Parse target subreddits from your CSV
target_subs = set()
with open(CSV_PATH, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        raw_str = row.get("subreddits", "")
        clean_str = (
            raw_str.replace("[", "")
            .replace("]", "")
            .replace("'", "")
            .replace('"', "")
        )
        for sub in clean_str.split(","):
            sub_name = sub.strip().lower()
            if sub_name:
                target_subs.add(sub_name)

# 3. Scan actual files on disk
downloaded_files = set()
for root, dirs, files in os.walk(TORRENT_DOWNLOAD_DIR):
    for filename in files:
        if filename.endswith(".zst"):
            downloaded_files.add(filename.lower())

# 4. Categorize each target subreddit
found_subs = {}
missing_subs = []

for sub in sorted(list(target_subs)):
    sub_comments = f"{sub}_comments.zst"
    sub_submissions = f"{sub}_submissions.zst"
    sub_single = f"{sub}.zst"

    has_comments = any(
        f.endswith(sub_comments) or f == sub_comments for f in downloaded_files
    )
    has_submissions = any(
        f.endswith(sub_submissions) or f == sub_submissions
        for f in downloaded_files
    )
    has_single = any(
        f.endswith(sub_single) or f == sub_single for f in downloaded_files
    )

    if has_comments or has_submissions or has_single:
        types = []
        if has_submissions:
            types.append("submissions")
        if has_comments:
            types.append("comments")
        if has_single:
            types.append("single archive")
        found_subs[sub] = ", ".join(types)
    else:
        missing_subs.append(sub)

# 5. Output Results Summary
print("========================================")
print(f" TOTAL TARGET SUBREDDITS: {len(target_subs)}")
print(f" FOUND ON DISK:           {len(found_subs)}")
print(f" MISSING FROM TORRENT:    {len(missing_subs)}")
print("========================================\n")

print("--- DOWNLOADED SUBREDDITS ---")
for sub, file_types in found_subs.items():
    print(f"  ✓ {sub} ({file_types})")

if missing_subs:
    print("\n--- MISSING SUBREDDITS (NOT IN TORRENT) ---")
    for sub in missing_subs:
        print(f"  ✗ {sub}")

 TOTAL TARGET SUBREDDITS: 121
 FOUND ON DISK:           115
 MISSING FROM TORRENT:    6

--- DOWNLOADED SUBREDDITS ---
  ✓ 2007scape (submissions, comments)
  ✓ 7daystodie (submissions, comments)
  ✓ amongus (submissions, comments)
  ✓ anno1800 (submissions, comments)
  ✓ apexlegends (submissions, comments)
  ✓ assettocorsa (submissions, comments)
  ✓ avatar (submissions, comments)
  ✓ balatro (submissions, comments)
  ✓ barotrauma (submissions, comments)
  ✓ beamng (submissions, comments)
  ✓ bindingofisaac (submissions, comments)
  ✓ blackdesertmobile (submissions, comments)
  ✓ blackdesertonline (submissions, comments)
  ✓ blackmythwukong (submissions, comments)
  ✓ brawlhalla (submissions, comments)
  ✓ btd6 (submissions, comments)
  ✓ crusaderkings (submissions, comments)
  ✓ cultofthelamb (submissions, comments)
  ✓ cuphead (submissions, comments)
  ✓ cyberpunkgame (submissions, comments)
  ✓ d (submissions, comments)
  ✓ darksouls (submissions, comments)
  ✓ darksouls3 (submissi

In [37]:
def get_torrents():

    print("Connecting to qBittorrent...")
    qbt_client = Client(host='localhost:8080')

    target_subs = set()

    with open(f"game_subreddits{date}.csv", "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:

            raw_str = row.get("subreddits", "")

            clean_str = (
                raw_str.replace("[", "")
                .replace("]", "")
                .replace("'", "")
                .replace('"', "")
            )


            for sub in clean_str.split(","):
                sub_name = sub.strip().lower()
                if sub_name:
                    target_subs.add(sub_name)


    torrents = qbt_client.torrents_info()
    reddit_torrent = next(
        (t for t in torrents if "reddit" in t.name.lower()), None
    )


    if not reddit_torrent:
        print("Could not find the Reddit dump torrent.")
        sys.exit(1)

    torrent_hash = reddit_torrent.hash
    files = qbt_client.torrents_files(torrent_hash=torrent_hash)
    file_ids_to_enable = []
    file_ids_to_disable = []

    for f in files:
        filename_lower = f.name.lower()
        base_name = filename_lower.split("/")[-1]
        is_target = any(
            base_name == f"{sub}_submissions.zst"
            or base_name == f"{sub}_comments.zst"
            or base_name == f"{sub}.zst"
            for sub in target_subs
        )

        file_id = f.id if hasattr(f, "id") else f.index

        if is_target:
            if f.priority == 0:
                file_ids_to_enable.append(file_id)
                print(f"  [QUEUED FOR DOWNLOAD] -> {f.name}")
            elif f.progress == 1.0:
                print(f"[KEEPING - ALREADY COMPLETED] -> {f.name}")
            else:
                print(f"[KEEPING - IN PROGRESS] -> {f.name}")
        else:
            if f.priority > 0 or f.progress > 0:
                file_ids_to_disable.append(file_id)
                print(f"[DISABLING/UNCHECKING] -> {f.name}")

    if file_ids_to_enable:
        print(
            f"\nEnabling {len(file_ids_to_enable)} file(s) for download/retention..."
        )
        qbt_client.torrents_file_priority(
            torrent_hash=torrent_hash, file_ids=file_ids_to_enable, priority=1
        )

    if file_ids_to_disable:
        print(f"Disabling {len(file_ids_to_disable)} unwanted file(s)...")
        qbt_client.torrents_file_priority(
            torrent_hash=torrent_hash, file_ids=file_ids_to_disable, priority=0
        )


In [38]:
get_torrents()

Connecting to qBittorrent...
  [QUEUED FOR DOWNLOAD] -> reddit/subreddits25/2007scape_comments.zst
  [QUEUED FOR DOWNLOAD] -> reddit/subreddits25/2007scape_submissions.zst
  [QUEUED FOR DOWNLOAD] -> reddit/subreddits25/7daystodie_comments.zst
  [QUEUED FOR DOWNLOAD] -> reddit/subreddits25/7daystodie_submissions.zst
[DISABLING/UNCHECKING] -> reddit/subreddits25/911archive_submissions.zst
[DISABLING/UNCHECKING] -> reddit/subreddits25/911dispatchers_comments.zst
[DISABLING/UNCHECKING] -> reddit/subreddits25/911dispatchers_submissions.zst
[DISABLING/UNCHECKING] -> reddit/subreddits25/911truth_comments.zst
[DISABLING/UNCHECKING] -> reddit/subreddits25/996_comments.zst
[DISABLING/UNCHECKING] -> reddit/subreddits25/996_submissions.zst
[DISABLING/UNCHECKING] -> reddit/subreddits25/99nightsintheforest_comments.zst
[DISABLING/UNCHECKING] -> reddit/subreddits25/99nightsintheforest_submissions.zst
[DISABLING/UNCHECKING] -> reddit/subreddits25/9anime_comments.zst
[DISABLING/UNCHECKING] -> reddit/su

In [ ]:
twitch_views_folder = f'2025_twitch_views'
os.makedirs(twitch_views_folder, exist_ok=True)
path = f'.\\2025_twitch_views\\'

for idx, game in df_filtered_games_reviews.iterrows():
    print(f"Collecting viewer_counts for {game['name']}...")
    twitch_views_temp = get_twitch_views(game['name'])
    twitch_views_temp_df = pd.DataFrame(twitch_views_temp)
    twitch_views_temp_df.to_csv(os.path.join(path,f"twitch_views_{game["appid"]}.csv"), index=False)
    time.sleep(1)

In [ ]:
appids = []

for idx, game in df_filtered_games_reviews.iterrows():
    appids.append(game['appid'])

appids_df = pd.DataFrame(appids)
appids_df.to_csv("appids.csv", index= False)

Here we execute the script for getting player counts.

python steamdb_downloader.py --file appids.csv --delay 100  

Here we execute the script for getting reviews

python steam_review_downloader.py appids.csv

To do: 
clean up code,
add markdown and comments,
review scripts,
change variable names?